# Health Insurance Premium Prediction — ML Training and Testing Report

This notebook trains and tests regression models using `data/insurance.csv`. It compares Linear Regression, Random Forest, and XGBoost (when installed), then saves the best model and a Markdown report.

> **Important:** This is an educational model. It must not be used for real insurance pricing, underwriting, eligibility, or other high-impact decisions.

## 1. Install packages (only if required)

In [ ]:
# Run once if packages are missing:
# %pip install pandas numpy matplotlib seaborn scikit-learn xgboost joblib

## 2. Import libraries and load data

In [ ]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

DATA_PATH = Path('data/insurance.csv')
MODEL_DIR = Path('models')
MODEL_DIR.mkdir(exist_ok=True)

data = pd.read_csv(DATA_PATH)
data.head()

## 3. Data quality and exploratory analysis

The original public dataset contains no income/salary field. The optional `salary` feature below is an educational proxy and should not be treated as real income data.

In [ ]:
print(f'Dataset shape: {data.shape}')
print('\nMissing values:')
display(data.isnull().sum().to_frame('missing_values'))
display(data.describe(include='all').T)

# Educational proxy, kept consistent with this project’s Flask application.
if 'salary' not in data.columns:
    data['salary'] = np.clip(
        180000 + data['age'] * 6500 + data['bmi'] * 2500 + data['children'] * 12000,
        100000, 10000000
    ).round(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(data=data, x='charges', hue='smoker', kde=True, ax=axes[0])
axes[0].set_title('Insurance Charges by Smoking Status')
sns.boxplot(data=data, x='smoker', y='charges', ax=axes[1])
axes[1].set_title('Charges Distribution by Smoking Status')
plt.tight_layout()

## 4. Prepare features and train/test split

In [ ]:
FEATURES = ['age', 'sex', 'bmi', 'children', 'smoker', 'region', 'salary']
TARGET = 'charges'
RANDOM_STATE = 42

data = data.dropna(subset=FEATURES + [TARGET]).copy()
X_train, X_test, y_train, y_test = train_test_split(
    data[FEATURES], data[TARGET], test_size=0.20, random_state=RANDOM_STATE
)
print(f'Training records: {len(X_train)}')
print(f'Testing records: {len(X_test)}')

## 5. Train and evaluate models

In [ ]:
def build_pipeline(regressor):
    preprocessor = ColumnTransformer([
        ('categorical', OneHotEncoder(handle_unknown='ignore'), ['sex', 'smoker', 'region']),
        ('numeric', 'passthrough', ['age', 'bmi', 'children', 'salary']),
    ])
    return Pipeline([('preprocessor', preprocessor), ('regressor', regressor)])

models = {
    'Linear Regression': build_pipeline(LinearRegression()),
    'Random Forest Regressor': build_pipeline(RandomForestRegressor(
        n_estimators=350, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1
    )),
}

try:
    from xgboost import XGBRegressor
    models['XGBoost Regressor'] = build_pipeline(XGBRegressor(
        n_estimators=400, learning_rate=0.05, max_depth=3,
        objective='reg:squarederror', random_state=RANDOM_STATE, n_jobs=-1
    ))
except ImportError:
    print('XGBoost is unavailable; skipped.')

trained_models, prediction_sets, results = {}, {}, []
for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)
    predictions = pipeline.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    trained_models[name] = pipeline
    prediction_sets[name] = predictions
    results.append({
        'Model': name,
        'MAE': round(mean_absolute_error(y_test, predictions), 2),
        'MSE': round(mse, 2),
        'RMSE': round(mse ** 0.5, 2),
        'R² Score': round(r2_score(y_test, predictions), 4),
    })

results_df = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
display(results_df)

## 6. Test-set visualization and sample predictions

In [ ]:
best_name = results_df.iloc[0]['Model']
best_predictions = prediction_sets[best_name]

plt.figure(figsize=(7, 6))
plt.scatter(y_test, best_predictions, alpha=0.7)
lower, upper = min(y_test.min(), best_predictions.min()), max(y_test.max(), best_predictions.max())
plt.plot([lower, upper], [lower, upper], 'r--', label='Perfect prediction')
plt.xlabel('Actual charges')
plt.ylabel('Predicted charges')
plt.title(f'Actual vs Predicted Charges — {best_name}')
plt.legend()
plt.show()

test_predictions = pd.DataFrame({
    'Actual Charge': y_test.to_numpy(),
    'Predicted Charge': best_predictions,
})
test_predictions['Absolute Error'] = abs(test_predictions['Actual Charge'] - test_predictions['Predicted Charge'])
display(test_predictions.head(10).round(2))

## 7. Save model and generate report

In [ ]:
best_model = trained_models[best_name]
joblib.dump(best_model, MODEL_DIR / 'premium_model.joblib')

metrics = {
    'dataset_rows': len(data),
    'training_rows': len(X_train),
    'testing_rows': len(X_test),
    'best_model': best_name,
    'results': results_df.to_dict(orient='records'),
}
(MODEL_DIR / 'evaluation_metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')

best_metrics = results_df.iloc[0]
report = f'''# ML Training and Testing Report

## Dataset
- Total records: {len(data)}
- Training records: {len(X_train)}
- Testing records: {len(X_test)}
- Target: charges

## Model comparison
```text
{results_df.to_string(index=False)}
```

## Selected model
{best_name} achieved the lowest MAE: {best_metrics['MAE']:,.2f}. Its test R² score is {best_metrics['R² Score']:.4f}.

## Limitation
This educational model is not suitable for real insurance underwriting or pricing decisions without robust validation, fairness testing, and regulatory review.
'''
(MODEL_DIR / 'training_testing_report.md').write_text(report, encoding='utf-8')

print(f'Best model saved: {MODEL_DIR / "premium_model.joblib"}')
print(f'Report saved: {MODEL_DIR / "training_testing_report.md"}')